# Train & Export Modelserver Artifacts (Spec 5 / US3)

Honest 3-way classifier comparison + BiomedBERT embedder export.

**Method (no leakage):**
- **70/15/15 stratified** split (seeded): `train` fits, `val` selects the winner, `test` is the
  single honest final number **and** becomes `modelserver/eval/eval_set.jsonl` (the CI gate set,
  never seen during selection).
- Report **macro-F1** (dataset is class-imbalanced).
- Candidates: **A** TF-IDF+LogReg (CPU, shippable), **B** BiomedBERT fine-tuned (GPU, shippable via
  ONNX), **C** GPT-4o zero-shot (**comparison baseline only** - an LLM cannot be the lean offline
  modelserver classifier: external dependency, non-determinism, PII egress, cost/latency).
- The **embedder** (BiomedBERT -> ONNX, quantized) is exported regardless of which classifier wins.

**SMOKE flag (cell below):** `SMOKE = True` runs a fast local sanity check (subsampled data, 1 epoch,
small C sample) and tags artifacts `-SMOKE` so they can't be shipped. Set `SMOKE = False` for the real
run (do that one on a **Colab T4 GPU** - BiomedBERT on CPU is ~a day/epoch).

**Requirements:** `uv sync --group training`. For Candidate C set `OPENAI_API_KEY` (terminal:
`$env:OPENAI_API_KEY=...` then launch Jupyter from that shell; Colab: Secrets panel). Never hardcode it.


In [6]:
import os, sys, json, hashlib, shutil
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42

# ---------------------------------------------------------------------------
# SMOKE = True  -> fast local sanity run: subsample, 1 epoch, tiny C sample.
#                 Artifacts are tagged '-SMOKE' (throwaway; eval gate will fail them).
# SMOKE = False -> real run: full data, 3 epochs. Use a Colab T4 GPU for this.
# ---------------------------------------------------------------------------
SMOKE = True
SMOKE_SFX = "-SMOKE" if SMOKE else ""
if SMOKE:
    print("=" * 64)
    print("SMOKE MODE ON: subsampled data + 1 epoch -> THROWAWAY artifacts.")
    print("Set SMOKE = False for the real run (full data, 3 epochs, GPU).")
    print("=" * 64)

print(sys.version)
import sklearn; print("sklearn", sklearn.__version__)
try:
    import torch, transformers, datasets
    print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
    print("transformers", transformers.__version__, "| datasets", datasets.__version__)
except Exception as e:
    print("training group not fully importable:", e)

# Resolve repo paths whether run from repo root or notebooks/
REPO = Path.cwd()
if not (REPO / "modelserver").exists():
    REPO = REPO.parent
MODELS_DIR = REPO / "modelserver" / "models"
EVAL_DIR = REPO / "modelserver" / "eval"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
print("MODELS_DIR =", MODELS_DIR)


SMOKE MODE ON: subsampled data + 1 epoch -> THROWAWAY artifacts.
Set SMOKE = False for the real run (full data, 3 epochs, GPU).
3.12.13 (main, Apr  7 2026, 20:53:22) [MSC v.1944 64 bit (AMD64)]
sklearn 1.9.0
torch 2.12.0+cpu | cuda: False
transformers 4.57.6 | datasets 5.0.0
MODELS_DIR = c:\Users\rasha\OneDrive\Desktop\SEF\AIE\final-project\pantera\modelserver\models


## 1. Load ADE Corpus v2 and inspect size / balance

In [7]:
from datasets import load_dataset

DATASET_NAME, DATASET_CONFIG = "ade_corpus_v2", "Ade_corpus_v2_classification"
ds = load_dataset(DATASET_NAME, DATASET_CONFIG)
print(ds)

df = pd.DataFrame(ds["train"])
df.columns = ["text", "label"]
df = df.dropna(subset=["text"]).reset_index(drop=True)
print("total examples:", len(df))
print("class balance:")
print(df["label"].value_counts())
print("positive rate:", round(df["label"].mean(), 4))


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 23516
    })
})
total examples: 23516
class balance:
label
0    16695
1     6821
Name: count, dtype: int64
positive rate: 0.2901


## 2. Stratified 70/15/15 split (train / val / test)

In [8]:
from sklearn.model_selection import train_test_split

# 70 / 15 / 15, stratified on the (imbalanced) label, fully seeded.
train_df, tmp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df["label"])
val_df, test_df = train_test_split(tmp_df, test_size=0.50, random_state=SEED, stratify=tmp_df["label"])

if SMOKE:  # tiny subsample just to prove the pipeline runs end-to-end
    train_df = train_df.sample(n=min(800, len(train_df)), random_state=SEED)
    val_df = val_df.sample(n=min(200, len(val_df)), random_state=SEED)
    test_df = test_df.sample(n=min(200, len(test_df)), random_state=SEED)

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(d), "pos_rate=%.3f" % d["label"].mean())

# Selection happens on val; test is touched ONLY for the final number + eval_set.jsonl.
results_val = {}   # candidate -> macro-F1 on val
results_test = {}  # candidate -> macro-F1 on test


train 800 pos_rate=0.287
val 200 pos_rate=0.275
test 200 pos_rate=0.235


## 3. Candidate A - TF-IDF + Logistic Regression (CPU, shippable)

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report

clf_a = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)),
    ("lr", LogisticRegression(C=4.0, class_weight="balanced", max_iter=1000, random_state=SEED)),
])
clf_a.fit(train_df["text"], train_df["label"])
results_val["A_tfidf_lr"] = f1_score(val_df["label"], clf_a.predict(val_df["text"]), average="macro")
print("Candidate A  val macro-F1:", round(results_val["A_tfidf_lr"], 4))
print(classification_report(val_df["label"], clf_a.predict(val_df["text"])))


Candidate A  val macro-F1: 0.8119
              precision    recall  f1-score   support

           0       0.90      0.90      0.90       145
           1       0.73      0.73      0.73        55

    accuracy                           0.85       200
   macro avg       0.81      0.81      0.81       200
weighted avg       0.85      0.85      0.85       200



## 4. Candidate B - BiomedBERT fine-tuned (GPU; shippable via ONNX)

Fine-tunes `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` for sequence
classification. Uses 1 epoch under SMOKE, 3 epochs for the real run. GPU strongly recommended.

In [10]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                            TrainingArguments, Trainer)
import torch
from torch.utils.data import Dataset

B_MODEL = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
MAX_LEN = 128
tok_b = AutoTokenizer.from_pretrained(B_MODEL)

class AdeDS(Dataset):
    def __init__(self, frame):
        self.enc = tok_b(list(frame["text"]), truncation=True, padding="max_length", max_length=MAX_LEN)
        self.labels = list(frame["label"])
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

def macro_f1_metric(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}

model_b = AutoModelForSequenceClassification.from_pretrained(B_MODEL, num_labels=2)
args = TrainingArguments(
    output_dir="/tmp/biomedbert-ade",
    num_train_epochs=1 if SMOKE else 3,
    per_device_train_batch_size=16, per_device_eval_batch_size=64,
    eval_strategy="epoch", save_strategy="no", seed=SEED, report_to="none",
    learning_rate=2e-5, warmup_ratio=0.1,
    fp16=torch.cuda.is_available(), dataloader_pin_memory=torch.cuda.is_available(),
)
trainer = Trainer(model=model_b, args=args,
                  train_dataset=AdeDS(train_df), eval_dataset=AdeDS(val_df),
                  compute_metrics=macro_f1_metric)
trainer.train()
val_pred_b = trainer.predict(AdeDS(val_df)).predictions.argmax(axis=1)
results_val["B_biomedbert"] = f1_score(val_df["label"], val_pred_b, average="macro")
print("Candidate B  val macro-F1:", round(results_val["B_biomedbert"], 4))


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Macro F1
1,No log,0.502367,0.572703


Candidate B  val macro-F1: 0.5727


## 5. Candidate C - GPT-4o zero-shot (comparison baseline only)

Reads `OPENAI_API_KEY` from the environment; **skips gracefully if absent**. Scores a fixed sample of
`val` (25 under SMOKE, 500 otherwise; set `C_SAMPLE = None` for the full split). Not shippable.

In [ ]:
# Colab: uncomment to pull the key from the Secrets panel into the environment.
# from google.colab import userdata; os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

C_MODEL = "gpt-4o-2024-08-06"
C_SAMPLE = 25 if SMOKE else 500  # set to None to score the entire val split (costs more)
PROMPT = ("Does the following sentence describe an adverse drug event (a harmful or "
          "unintended reaction to a medication)? Answer with exactly one word: YES or NO.\n\n{text}")

# Key from env (set in the launching shell) or an interactive prompt that does not echo and is
# never stored in the notebook source. Press Enter at the prompt to skip Candidate C.
import getpass
api_key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OPENAI_API_KEY (Enter to skip C): ")
if not api_key:
    print("No OPENAI_API_KEY provided - skipping Candidate C (comparison-only).")
else:
    from openai import OpenAI
    # Bound every request so a network/rate-limit issue fails fast instead of hanging the cell.
    client = OpenAI(api_key=api_key, timeout=30.0, max_retries=2)
    c_eval = val_df.sample(n=min(C_SAMPLE, len(val_df)), random_state=SEED) if C_SAMPLE else val_df
    preds_c, gold_c, errors = [], [], 0
    for n, (_, row) in enumerate(c_eval.iterrows(), 1):
        try:
            r = client.chat.completions.create(
                model=C_MODEL, temperature=0, max_tokens=5,
                messages=[{"role": "user", "content": PROMPT.format(text=row["text"])}],
            )
            ans = (r.choices[0].message.content or "").strip().upper()
            preds_c.append(1 if ans.startswith("YES") else 0)
            gold_c.append(int(row["label"]))
        except Exception as e:
            errors += 1
            if errors <= 3:  # surface the first few so you can see WHY (401 key, 429 rate, model name...)
                print("  call %d failed: %s: %s" % (n, type(e).__name__, e))
        if n % 10 == 0:
            print("  ...%d/%d done" % (n, len(c_eval)))
    if gold_c:
        results_val["C_gpt4o_sample"] = f1_score(gold_c, preds_c, average="macro")
        print("Candidate C  val macro-F1 (n=%d, %d errors):" % (len(gold_c), errors),
              round(results_val["C_gpt4o_sample"], 4))
    else:
        print("Candidate C: all %d calls failed - check key / network / model name. Skipping." % errors)

## 6. Selection on validation (winner = best *shippable* candidate)

In [12]:
cmp = pd.DataFrame(
    [{"candidate": k, "val_macro_f1": round(v, 4)} for k, v in results_val.items()]
).sort_values("val_macro_f1", ascending=False)
print(cmp.to_string(index=False))

SHIPPABLE = {"A_tfidf_lr", "B_biomedbert"}  # C (LLM) is comparison-only
winner = max(SHIPPABLE & set(results_val), key=lambda k: results_val[k])
print("\nWinner (shippable):", winner, "val macro-F1:", round(results_val[winner], 4))
if "C_gpt4o_sample" in results_val and results_val["C_gpt4o_sample"] > results_val[winner]:
    print("Note: GPT-4o scored higher but is not deployable as the lean offline classifier.")


   candidate  val_macro_f1
  A_tfidf_lr        0.8119
B_biomedbert        0.5727

Winner (shippable): A_tfidf_lr val macro-F1: 0.8119


## 7. Final honest number on the held-out test split

In [13]:
if winner == "A_tfidf_lr":
    test_pred = clf_a.predict(test_df["text"])
else:
    test_pred = trainer.predict(AdeDS(test_df)).predictions.argmax(axis=1)
results_test[winner] = f1_score(test_df["label"], test_pred, average="macro")
print("WINNER", winner, "TEST macro-F1: %.4f (CI gate needs >= 0.80)" % results_test[winner])
if SMOKE:
    print("(SMOKE run - this number is meaningless; flip SMOKE=False for the real one.)")
print(classification_report(test_df["label"], test_pred))


WINNER A_tfidf_lr TEST macro-F1: 0.7533 (CI gate needs >= 0.80)
(SMOKE run - this number is meaningless; flip SMOKE=False for the real one.)
              precision    recall  f1-score   support

           0       0.89      0.88      0.88       153
           1       0.61      0.64      0.62        47

    accuracy                           0.82       200
   macro avg       0.75      0.76      0.75       200
weighted avg       0.82      0.82      0.82       200



## 8. Export the winning classifier (joblib for A, quantized ONNX for B)

In [ ]:
import joblib

if winner == "A_tfidf_lr":
    joblib.dump(clf_a, MODELS_DIR / "classifier.joblib")
    clf_file, clf_fmt, clf_ver = "classifier.joblib", "joblib", "v1.0-tfidf-lr" + SMOKE_SFX
    (MODELS_DIR / "classifier.onnx").unlink(missing_ok=True)  # keep manifest/serving consistent
    print("Exported classifier.joblib")
else:
    from optimum.onnxruntime import ORTModelForSequenceClassification, ORTQuantizer
    from optimum.onnxruntime.configuration import AutoQuantizationConfig
    FT_DIR = Path("/tmp/biomedbert-ade-final"); FT_DIR.mkdir(exist_ok=True)
    CLF_Q = Path("/tmp/biomedbert-ade-q"); CLF_Q.mkdir(exist_ok=True)
    trainer.save_model(FT_DIR); tok_b.save_pretrained(FT_DIR)
    ORTModelForSequenceClassification.from_pretrained(FT_DIR, export=True).save_pretrained(FT_DIR)
    ORTQuantizer.from_pretrained(FT_DIR).quantize(
        save_dir=CLF_Q,
        quantization_config=AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False),
    )
    # Move ONLY the quantized model into models/. os.replace overwrites (Path.rename raises on
    # Windows if the target exists); quantizing into a temp dir avoids optimum's config clutter.
    os.replace(CLF_Q / "model_quantized.onnx", MODELS_DIR / "classifier.onnx")
    (MODELS_DIR / "classifier.joblib").unlink(missing_ok=True)
    clf_file, clf_fmt, clf_ver = "classifier.onnx", "onnx", "v1.0-biomedbert-quant" + SMOKE_SFX
    print("Exported classifier.onnx (quantized)")

## 9. Export the BiomedBERT embedder to ONNX (quantized) + tokenizer

Exported regardless of the classifier winner. **Serving-compatibility check:** the modelserver
feeds only `input_ids` + `attention_mask` (no `token_type_ids`). The cell below verifies the
exported ONNX runs with exactly that input set, so a signature mismatch surfaces here, not in prod.

In [ ]:
from optimum.onnxruntime import ORTModelForFeatureExtraction, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

EMB_MODEL = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
TMP_EMB = Path("/tmp/embedder_onnx"); TMP_EMB.mkdir(exist_ok=True)
TMP_EMB_Q = Path("/tmp/embedder_onnx_q"); TMP_EMB_Q.mkdir(exist_ok=True)
ORTModelForFeatureExtraction.from_pretrained(EMB_MODEL, export=True).save_pretrained(TMP_EMB)
ORTQuantizer.from_pretrained(TMP_EMB).quantize(
    save_dir=TMP_EMB_Q,
    quantization_config=AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False),
)
# Move ONLY the quantized model into models/. os.replace overwrites (Path.rename raises on Windows
# if the target exists); quantizing into a temp dir avoids optimum's config.json/ort_config.json.
os.replace(TMP_EMB_Q / "model_quantized.onnx", MODELS_DIR / "embedder.onnx")

from transformers import AutoTokenizer
AutoTokenizer.from_pretrained(EMB_MODEL).save_pretrained(TMP_EMB)
shutil.copy(TMP_EMB / "tokenizer.json", MODELS_DIR / "tokenizer.json")
print("embedder.onnx %.1f MB" % (os.path.getsize(MODELS_DIR / "embedder.onnx") / 1e6))

# Serving-compatibility check.
# The modelserver feeds input_ids + attention_mask; token_type_ids (all-zeros for single-sentence
# BERT) is supplied automatically when the ONNX graph needs it (embedder.py detects it at runtime).
import onnxruntime as ort
from tokenizers import Tokenizer
tk = Tokenizer.from_file(str(MODELS_DIR / "tokenizer.json"))
enc = tk.encode("warfarin associated hepatotoxicity")
ids = np.array([enc.ids], dtype=np.int64); mask = np.array([enc.attention_mask], dtype=np.int64)
sess = ort.InferenceSession(str(MODELS_DIR / "embedder.onnx"), providers=["CPUExecutionProvider"])
need = {i.name for i in sess.get_inputs()}
allowed = {"input_ids", "attention_mask", "token_type_ids"}  # serving supplies all three as needed
unknown = need - allowed
assert not unknown, "ONNX needs unexpected inputs %s - check export" % unknown
feed = {"input_ids": ids, "attention_mask": mask}
if "token_type_ids" in need:
    feed["token_type_ids"] = np.zeros_like(ids)
    print("Note: ONNX needs token_type_ids (all-zeros) - serving code supplies it automatically.")
out = sess.run(None, {k: v for k, v in feed.items() if k in need})
print("embedder OK; output shape:", out[0].shape, "(expect [...,768])")

## 10. Write manifest.json (SHA-256 stamps) and the eval_set (= test split)

In [ ]:
def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

manifest = {"artifacts": [
    {"name": "classifier", "file": clf_file, "format": clf_fmt, "version": clf_ver,
     "sha256": sha256(MODELS_DIR / clf_file)},
    {"name": "embedder", "file": "embedder.onnx", "format": "onnx",
     "version": "v1.0-biomedbert-quant" + SMOKE_SFX,
     "sha256": sha256(MODELS_DIR / "embedder.onnx"), "dim": 768, "max_tokens": 512},
    {"name": "tokenizer", "file": "tokenizer.json", "format": "tokenizer",
     "version": "v1.0-biomedbert" + SMOKE_SFX, "sha256": sha256(MODELS_DIR / "tokenizer.json")},
]}
(MODELS_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

# eval_set.jsonl = the held-out TEST split (never used for selection) -> honest CI gate.
with (EVAL_DIR / "eval_set.jsonl").open("w") as f:
    for _, row in test_df.iterrows():
        f.write(json.dumps({"text": row["text"], "label": int(row["label"])}) + "\n")
print("wrote eval_set.jsonl:", len(test_df), "examples")
if SMOKE:
    print("\nSMOKE artifacts written (tagged -SMOKE). DO NOT commit/ship; re-run with SMOKE=False.")


## Summary

1. Selection on **val**, final number on **test** (held out) -> no leakage; `eval_set.jsonl` is the
   test split, so the CI gate is honest.
2. All three candidates compared; the shipped one is the best **deployable** model (A or B). GPT-4o is
   a baseline only.
3. **SMOKE check:** if you ran with `SMOKE = True`, artifacts/versions are tagged `-SMOKE` and the
   numbers are meaningless - flip to `SMOKE = False` and re-run (on a Colab T4) for the real artifacts.
4. Artifacts in `modelserver/models/`. Then, on a machine with the repo:

```
uv run python modelserver/eval/run_eval.py          # must print PASS (macro-F1 >= 0.80)
# If embedder.onnx or classifier.onnx > 100 MB: git lfs track before committing (see docs/RUNBOOK.md)
docker compose build modelserver api                 # api ships tokenizer.json (FR-025)
docker compose up -d --wait modelserver api
```

Swapping the embedder changes its SHA-256 -> existing chunks must be re-indexed. Do this **before**
indexing for Spec 7 (the index is currently empty).
